In [22]:
import re
import json

In [161]:
class BPEMethod:
    def __init__(self, file_path: str):
        with open(file_path) as file:
            self.text = self.format_text(file.read())
            self.tokens = list(self.text)
            self.codes = {}

    def format_text(self, text: str):
        text = text.lower()
        text = re.sub(r'[^а-яёa-z0-9 ,.!?-]', '', text)
        text = re.sub(r' +', ' ', text)

        return text

    def get_counts(self):
        counts = {}
        for a, b in zip(self.tokens, self.tokens[1:]):
            key = (a, b)
            counts[key] = counts.get(key, 0) + 1
        return counts
            
    def check_bytes(self, codes: dict):
        serializable_codes = {str(k): list(v) for k, v in codes.items()}
        json_string = json.dumps(serializable_codes, separators=(',', ':'), ensure_ascii=False)
        return len(json_string.encode('utf-8')) <= 1024

    def replace_pair(self, index: str, pair: tuple[str]):
        new_tokens = []
        i = 0
        while i < len(self.tokens):
            if i < len(self.tokens) - 1 and (self.tokens[i], self.tokens[i+1]) == pair:
                new_tokens.append(index)
                i += 2
            else:
                new_tokens.append(self.tokens[i])
                i += 1
        self.tokens = new_tokens

    def bpe_compress(self):

        next_token_id = 0

        while True:
            counts = self.get_counts()
            if not counts:
                break
                
            max_pair = max(counts, key=counts.get)
            if counts[max_pair] <= 1:
                break

            test_codes = self.codes.copy()
            token_str = f'T{next_token_id}'
            test_codes[token_str] = max_pair

            if not self.check_bytes(test_codes):
                break

            self.codes[token_str] = max_pair
            self.replace_pair(token_str, max_pair)
            next_token_id += 1
        
        return ''.join(self.tokens)

    def bpe_decompress(self):

        current_tokens = self.tokens.copy()

        while any(token.startswith('T') for token in current_tokens):
            new_tokens = []
            for token in current_tokens:
                if token.startswith('T'):
                    pair = self.codes[token]
                    new_tokens.extend(pair)
                else:
                    new_tokens.append(token)
            current_tokens = new_tokens
        return ''.join(current_tokens)

In [162]:
test = BPEMethod('text.txt')
print(test.bpe_compress()[:1000])

print('-' * 30)

print(test.bpe_decompress()[:1000])

михаT22 булгакT6.T10кT6ыT5яйцагT42вT41.курT35кулюмT12T43эT1T10фT29соT11T57сT38T6а16 апрT16T261928 гT46аT0вечT9омT49T10фT29сT45 зоT34огиT3iv гT56ударT7вT13T20T41унивT9сT43T23T4T3дирекT18р зооT39T7T43утT4T50мT56квеT57сT38T6T12ошT16T12T19воT25T17бT39T23T49омещающийсяT12 зооT39T7T43утеT0чT36нT4улицT5гT9цT13а.T1T10фT29сT45 T30жегT12T9хниT25мT53T6ыT25шар T3оглядT16ся.T31чT8T2ужасающеT25T17таT7T10фыT14ужT51счT43T53T37зT8ожT13T27T44имT13T51T50эT18т злT56чаT7T27йT12ечT9T0T11вT51T17кT57вопT35чиT20ю эT18T25T17таT7T10фыT19ледуT23T19чT43T53T37имT13T20T1T10фT29сT45T4вT42димирT4ипT53ьевичаT57сT38T6а.T32T47T48лT2рT6T5158 лT23T28гT34T6T4T30мечT53T16ьT31яT0T18лT17чомT0лысаяT0сT1учT17мT3жT16тT6T53ыхT12T34T56T0T18рчащимиT1T2боT17мT28лицT2гT42дкT2T54бT35T18еT0нижняT26губT4T54пячT13T4впT9едT28T33 эT18гоT57сT38T6сT15T5лицT2вечT51T20сT22T2нT4себT5нT29T15лькT2T17пT35зT27T25T33печаT18к.T14T4кT11сT20мT14T56T47T7аT10мT46T27T5мT8T13ькиT5очкT3вT19T9ебряT20T25опT11веT0гT42зкT3блеT7ящиеT0небT34ьшиеT0T10T7уT12ысоT15го